# Replication of Buehler et al. (2019), *Deep Hedging* (Heston case)

Buehler, Gonon, Teichmann and Wood replace the analytic hedging rule with a
network trained to minimise a convex risk measure of the terminal P&L. This
notebook runs the Heston experiment with proportional transaction costs and
checks the paper's own correctness test.

The test that matters most is the frictionless one: with **zero** costs, a
correct hedger should rediscover something close to the Black-Scholes delta. If
it does not, nothing else it reports can be trusted, so it comes first.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import torch

from hedging import bench as B, bs, deep, metrics
from hedging.hyper import Hyper

BASE = B.BenchConfig()
print("market:", BASE.model, "cost:", BASE.cost, "premium:", round(BASE.premium, 4))

## 1. The frictionless check

Train with `cost = 0` and compare the positions the network holds with the
Black-Scholes delta at the same state. Correlation near one and a slope near
one would mean the network recovered the analytic hedge. The `alpha` of the
CVaR objective controls how much weight the tail gets; the mean objective
(`alpha = 1`) is expected to fail this check because with a volatility premium
it can simply refuse to hedge, which is exactly the failure mode the main
benchmark was redesigned to avoid.


In [ ]:
from dataclasses import replace

cfg0 = replace(BASE, cost=0.0, n_val_paths=5_000)
hyper = Hyper(epochs=60, hidden=64, depth=3, alpha=0.05, torch_seeds=(0,))
paths = cfg0.simulate(cfg0.model, 2001, 5_000)
tensor = torch.as_tensor(paths, dtype=torch.float32)

nets, _ = B.train_ensemble(cfg0, hyper)
learned = deep.hedge_positions(tensor, nets[0], cfg0.K, cfg0.dt, cfg0.T)

n_steps = paths.shape[1] - 1
target = np.stack([
    bs.bs_delta(paths[:, j], cfg0.K, max(cfg0.T - j * cfg0.dt, 0.0), 0.0, cfg0.sigma_impl, "call")
    for j in range(n_steps)
], axis=1)
x, y = learned.reshape(-1), target.reshape(-1)
print("correlation:", round(float(np.corrcoef(x, y)[0, 1]), 4))
print("slope:", round(float(np.polyfit(y, x, 1)[0]), 3))
print("rmse:", round(float(np.sqrt(np.mean((x - y) ** 2))), 4))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y, x, s=3, alpha=0.15)
lims = [0, 1]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel("Black-Scholes delta")
ax.set_ylabel("learned position")
ax.set_title("Frictionless hedger vs delta")
plt.show()

## 2. Heston with transaction costs

With costs, the comparison is against the plain Black-Scholes delta at every
step (Buehler's benchmark) and against the frozen cost-aware band baseline used
in the rest of the project. The learned hedger is the objective from
`hedging/deep.py`: CVaR of the terminal P&L, which is the tail objective the
main benchmark settled on.


In [ ]:
costs = [0.0, 0.002, 0.005, 0.01]
hyper = Hyper(epochs=60, hidden=64, depth=3, alpha=0.65, torch_seeds=(0, 1, 2))

records = []
for cost in costs:
    cfg = replace(BASE, cost=cost)
    nets, _ = B.train_ensemble(cfg, hyper)
    learned = B.NetworkStrategy(cfg, nets)
    delta = B.ClassicalStrategy(cfg, {"strategy": "time", "frequency": 1, "delta_band": None, "price_band": None})
    band = B.ClassicalStrategy(cfg, B.load_classical_best())
    deep_s, delta_s, band_s = [], [], []
    for seed in cfg.val_seeds:
        paths = cfg.simulate(cfg.model, seed, cfg.n_val_paths)
        deep_s.append(metrics.sharpe(learned.pnl(paths)))
        delta_s.append(metrics.sharpe(delta.pnl(paths)))
        band_s.append(metrics.sharpe(band.pnl(paths)))
    records.append({
        "cost_bp": cost * 1e4,
        "learned": np.mean(deep_s),
        "bs_delta": np.mean(delta_s),
        "frozen_band": np.mean(band_s),
    })
table = pl.DataFrame(records)
table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for key, label in (("learned", "learned"), ("bs_delta", "BS delta"), ("frozen_band", "frozen band")):
    ax.plot(table["cost_bp"], table[key], marker="o", label=label)
ax.set_xlabel("transaction cost (bp)")
ax.set_ylabel("Sharpe ratio")
ax.set_title("Heston: learned versus classical hedging")
ax.legend()
plt.show()

### Reading the result

- At zero cost the learned hedger tracks the delta closely but does not match
  its Sharpe exactly, because the CVaR objective weights the tail rather than
  the variance. That is expected and is the reason the frictionless check is
  reported as a correlation and a slope, not as an equality.
- As the cost rises, the learned hedger's advantage over the fixed delta
  changes; the frozen band is a strong comparator because it was tuned on its
  own pool for a cost aware regime.
- Run-to-run variance is real: every number here averages three training seeds,
  and a difference smaller than the spread across those seeds is not a result.
  The main benchmark reports the per-seed edge and its minimum for exactly this
  reason.
